# Phase 1 — Encoder-Only Legal Language Model (MLM Pre-training)

**Local training** on any machine with a CUDA GPU (T4 16GB or better).

Prerequisites:
- `bpe_40k.json` and `a1_tokenizer.py` in the same directory as this notebook.

This notebook:
1. Loads the pre-trained BPE tokenizer
2. Downloads, cleans, and tokenizes the corpus (temperature-scaled mixture, α=0.5)
3. Builds document-preserving train/val blocks with 15% MLM masking and fixed validation masks
4. Defines the Encoder Transformer (RoPE, RMSNorm, SwiGLU, weight tying)
5. Trains with mixed-precision + gradient accumulation
6. Evaluates token-weighted validation loss and saves resumable checkpoints plus the best model
7. Demonstrates MLM fill-in predictions

**Architecture:** ~125M parameters, 14-layer encoder, bidirectional attention, MLM objective.

**Important setup notes**
- `bpe_40k.json` and `a1_tokenizer.py` must be in the notebook directory.
- If the dataset requires Hugging Face authentication, set `HF_TOKEN` in your environment; do not hardcode a token in this notebook.
- The notebook adds `[MASK]` as a model-only ID immediately after the tokenizer vocabulary.
- Token blocks are written to disk and are document-preserving; the final block of each document is padded and masked from attention/loss.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ijson", "tqdm", "huggingface_hub"])

import os, sys, re, json, html, unicodedata, time, math, random
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import ijson
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1 — Configuration (all hyperparameters in one place)

In [ ]:
@dataclass
class Config:
    # ── Model ──
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 14        # ~125M params (with weight tying)
    n_heads: int         = 12
    d_ff: int            = 2048      # SwiGLU intermediate
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10_000.0

    # ── MLM ──
    mask_prob: float     = 0.15
    mask_token_id: int   = 3         # set after tokenizer loads
    random_token_prob: float = 0.10  # of the 15%: 80% [MASK], 10% random, 10% keep
    keep_token_prob: float   = 0.10

    # ── Training ──
    lr: float            = 1e-4
    min_lr: float        = 1e-5
    warmup_steps: int    = 1_000
    max_steps: int       = 30_000
    batch_size: int      = 16
    grad_accum: int      = 2        # effective batch = 32
    weight_decay: float  = 0.01
    max_grad_norm: float = 1.0
    betas: tuple         = (0.9, 0.999)
    use_amp: bool        = True

    # ── Data ──
    shares: dict         = None
    val_frac: float      = 0.02

    # ── Logging / checkpointing ──
    log_interval: int    = 50
    eval_interval: int   = 500
    save_interval: int   = 2_000
    patience: int        = 5
    seed: int            = 42

    def __post_init__(self):
        if self.shares is None:
            # ── Temperature-scaled sampling (Conneau et al., 2020 — XLM-R) ──
            NATURAL = {"High Courts": 0.551, "Supreme Court": 0.073,
                       "Tribunals": 0.335, "Acts": 0.041}
            ALPHA = 0.5
            raw = {s: p ** ALPHA for s, p in NATURAL.items()}
            t = sum(raw.values())
            self.shares = {s: round(v / t, 2) for s, v in raw.items()}
            diff = round(1.0 - sum(self.shares.values()), 2)
            self.shares[max(self.shares, key=self.shares.get)] += diff
        self.head_dim = self.d_model // self.n_heads
        assert self.d_model % self.n_heads == 0

    @property
    def eff_batch(self): return self.batch_size * self.grad_accum

CFG = Config()

# ── Paths ──
HF_REPO  = "Exploration-Lab/CS-781-Capstone"
# Optional: set HF_TOKEN in your shell/environment if this dataset requires authentication.
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_FILES = {
    "Acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
OUT       = Path("./output")
OUT.mkdir(exist_ok=True)
CKPT_DIR  = OUT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)

print(f"Config: ~{CFG.d_model}d x {CFG.n_layers}L x {CFG.n_heads}H, "
      f"eff_batch={CFG.eff_batch}, max_steps={CFG.max_steps}")
print(f"MLM mask_prob={CFG.mask_prob:.0%}")
print(f"Mixture (alpha=0.5): {CFG.shares}")

## 2 — Load tokenizer

In [ ]:
# Tokenizer files should be in the same directory as this notebook
sys.path.insert(0, ".")
from a1_tokenizer import BPETokenizer

tok_path = Path("bpe_40k.json")
assert tok_path.exists(), (
    "bpe_40k.json not found! Place it in the same directory as this notebook."
)

tokenizer = BPETokenizer.load(str(tok_path))

# Existing tokenizer vocabulary is unchanged. Reserve one additional ID for model-only [MASK].
BASE_VOCAB_SIZE = int(tokenizer.vocab_size)
MASK_ID = BASE_VOCAB_SIZE
CFG.mask_token_id = MASK_ID
CFG.vocab_size = BASE_VOCAB_SIZE + 1

# This tokenizer implementation is expected to expose special_tokens as {token: id}.
SPECIAL_TOKEN_MAP = getattr(tokenizer, "special_tokens", {})
if not isinstance(SPECIAL_TOKEN_MAP, dict):
    SPECIAL_TOKEN_MAP = {}

PAD_ID = SPECIAL_TOKEN_MAP.get("[PAD]")
if PAD_ID is None:
    raise ValueError(
        "Tokenizer must expose its [PAD] ID in tokenizer.special_tokens. "
        "Inspect a1_tokenizer.py and set PAD_ID explicitly if its API differs."
    )
PAD_ID = int(PAD_ID)

BASE_SPECIAL_IDS = {int(v) for v in SPECIAL_TOKEN_MAP.values()}
BASE_SPECIAL_IDS.add(MASK_ID)
RANDOM_TOKEN_IDS = np.asarray(
    [i for i in range(BASE_VOCAB_SIZE) if i not in BASE_SPECIAL_IDS],
    dtype=np.int64,
)
if len(RANDOM_TOKEN_IDS) == 0:
    raise ValueError("No ordinary vocabulary IDs available for random-token replacement.")

# Smoke test
_test = "The Supreme Court held that the appellant was entitled to relief."
assert tokenizer.decode(tokenizer.encode(_test)) == _test, "Round-trip failed"

print(f"Tokenizer loaded: base_vocab={BASE_VOCAB_SIZE:,}")
print(f"PAD_ID={PAD_ID}; MASK_ID={MASK_ID}; model_vocab={CFG.vocab_size:,}")
print("Tokenizer smoke test passed")

## 3 — Data download + cleaning utilities

In [ ]:
LOCAL = {}
for src, fn in HF_FILES.items():
    LOCAL[src] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"{src:14s} {Path(LOCAL[src]).stat().st_size / 2**30:.2f} GB")

# ── Cleaning (same as tokenizer notebook) ──
TAG_RE   = re.compile(r"</?[A-Za-z][^<>]{0,200}>")
URL_RE   = re.compile(r"https?://\S+|www\.\S+", re.I)
ENT_RE   = re.compile(r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);", re.I)
CTRL_RE  = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WSP_RE   = re.compile(r"[\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000]")
ZW_RE    = re.compile(r"[\u200b\u200c\u200d\u2060]")
SPACE_RE = re.compile(r"\s+")
MAX_REPL = 50
MIN_CHARS = 50

def clean_text(text):
    if text is None: return ""
    if isinstance(text, list): text = " ".join(map(str, text))
    elif isinstance(text, dict): text = " ".join(map(str, text.values()))
    else: text = str(text)
    if text.count("\ufffd") > MAX_REPL: return ""
    text = TAG_RE.sub(" ", text)
    text = html.unescape(text)
    text = URL_RE.sub(" ", text)
    text = ENT_RE.sub(" ", text)
    text = text.replace("\ufeff", " ").replace("\ufffd", " ")
    text = CTRL_RE.sub(" ", text)
    text = WSP_RE.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    text = ZW_RE.sub("", text)
    return SPACE_RE.sub(" ", text).strip()

TEXT_KEYS = ("text", "content", "judgment", "document", "body")
def extract_text(rec):
    if isinstance(rec, str): return rec
    if isinstance(rec, dict):
        for k in TEXT_KEYS:
            v = rec.get(k)
            if isinstance(v, str): return v
            if isinstance(v, list): return " ".join(map(str, v))
        return None
    return None

def iter_records(path):
    with open(path, "rb") as f:
        first = f.read(4096).lstrip()[:1]
    with open(path, "rb") as f:
        if first == b"[": yield from ijson.items(f, "item")
        elif first == b"{":
            for _, v in ijson.kvitems(f, ""): yield v
        else: raise ValueError(path)

## 4 — Estimate corpus budget (bottleneck-driven) → stream & tokenize

Same bottleneck-driven budget as the decoder notebook. Targets ~500M clean
tokens for the encoder (~125M params).

In [ ]:
# ── 4a: Estimate corpus budget ──
CLEAN_RATIO = 0.85
sizes = {s: Path(p).stat().st_size for s, p in LOCAL.items()}
est_chars = {s: int(sizes[s] * CLEAN_RATIO) for s in sizes}

budgets_if = {src: est_chars[src] / CFG.shares[src] for src in CFG.shares}
bottleneck = min(budgets_if, key=budgets_if.get)
TARGET_CHARS = int(budgets_if[bottleneck])

print("Estimated cleaned chars per source:")
for src in sizes:
    print(f"  {src:14s} {est_chars[src]/1e9:.2f}B "
          f"(share={CFG.shares[src]:.0%}, budget_if_bottleneck={budgets_if[src]/1e9:.2f}B)")
print(f"\nBottleneck source: {bottleneck}")
print(f"TARGET_CHARS: {TARGET_CHARS:,} ({TARGET_CHARS/1e9:.2f}B)")

# ── 4b: Stream documents into fixed-length blocks, preserving document boundaries ──
# Each document is chunked independently. The final partial chunk is PAD-filled.
# Train/validation splitting happens at document level, so chunks from one document
# cannot leak across the two splits.
BLOCK_DIR = OUT / "token_blocks"
BLOCK_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_BIN = BLOCK_DIR / "train_blocks.i32"
VAL_BIN = BLOCK_DIR / "val_blocks.i32"
for fp in (TRAIN_BIN, VAL_BIN):
    if fp.exists():
        fp.unlink()

def write_document_blocks(file_obj, ids, block_len, pad_id):
    ids = np.asarray(ids, dtype=np.int32)
    if ids.size == 0:
        return 0
    n_written = 0
    for start in range(0, len(ids), block_len):
        chunk = ids[start:start + block_len]
        if len(chunk) < block_len:
            chunk = np.pad(chunk, (0, block_len - len(chunk)), constant_values=pad_id)
        file_obj.write(chunk.tobytes())
        n_written += 1
    return n_written

rng = random.Random(CFG.seed)
stats = {}
train_blocks = val_blocks = 0

print("\nStreaming, cleaning, tokenizing, and writing blocks...")
with open(TRAIN_BIN, "wb") as train_f, open(VAL_BIN, "wb") as val_f:
    for src, path in LOCAL.items():
        budget = TARGET_CHARS * CFG.shares[src]
        p = min(1.0, budget / max(CLEAN_RATIO * sizes[src], 1))
        chars = docs = train_toks = val_toks = 0
        src_train_blocks = src_val_blocks = 0

        for rec in tqdm(iter_records(path), desc=src):
            if p < 1.0 and rng.random() > p:
                continue
            raw = extract_text(rec)
            if raw is None:
                continue
            text = clean_text(raw)
            if len(text) < MIN_CHARS:
                continue

            ids = tokenizer.encode(text, add_special_tokens=True)
            if not ids:
                continue

            if rng.random() < CFG.val_frac:
                val_toks += len(ids)
                src_val_blocks += write_document_blocks(
                    val_f, ids, CFG.max_seq_len, PAD_ID
                )
            else:
                train_toks += len(ids)
                src_train_blocks += write_document_blocks(
                    train_f, ids, CFG.max_seq_len, PAD_ID
                )

            chars += len(text)
            docs += 1
            if chars >= budget * 1.05:
                break

        train_blocks += src_train_blocks
        val_blocks += src_val_blocks
        stats[src] = {
            "docs": docs, "chars": chars, "train_toks": train_toks,
            "val_toks": val_toks, "train_blocks": src_train_blocks,
            "val_blocks": src_val_blocks,
        }
        print(
            f"  {src}: {docs:,} docs, {chars/1e6:.1f}M chars, "
            f"{train_toks:,} train toks, {val_toks:,} val toks, "
            f"{src_train_blocks:,} train blocks, {src_val_blocks:,} val blocks "
            f"(sampling p={p:.2%})"
        )

if train_blocks == 0 or val_blocks == 0:
    raise RuntimeError(
        f"Not enough blocks: train={train_blocks}, val={val_blocks}. "
        "Check corpus paths, tokenizer, and validation fraction."
    )

# Read-only memory maps avoid loading all token IDs into RAM.
train_arr = np.memmap(
    TRAIN_BIN, dtype=np.int32, mode="r",
    shape=(train_blocks, CFG.max_seq_len)
)
val_arr = np.memmap(
    VAL_BIN, dtype=np.int32, mode="r",
    shape=(val_blocks, CFG.max_seq_len)
)

tot_c = sum(s["chars"] for s in stats.values())
print("\nRealised source mix:")
for src, s in stats.items():
    print(f"  {src:14s} {s['chars']/max(tot_c, 1):.1%} (target {CFG.shares[src]:.0%})")
print(f"\nTrain blocks: {train_blocks:,}; validation blocks: {val_blocks:,}")
print(f"Disk-backed block files: {TRAIN_BIN} ({TRAIN_BIN.stat().st_size/1e9:.2f} GB), "
      f"{VAL_BIN} ({VAL_BIN.stat().st_size/1e9:.2f} GB)")

## 5 — MLM Dataset & DataLoader

In [ ]:
class MLMDataset(Dataset):
    """Document-preserving fixed-length blocks; random masks for train, fixed masks for validation."""
    def __init__(self, data, block_len, cfg, pad_id, special_ids, random_token_ids, is_train, seed=42):
        self.data = data
        self.block_len = block_len
        self.cfg = cfg
        self.pad_id = int(pad_id)
        self.special_ids = set(int(i) for i in special_ids)
        self.random_token_ids = random_token_ids
        self.is_train = is_train
        self.seed = int(seed)

    def __len__(self):
        return int(self.data.shape[0])

    def __getitem__(self, idx):
        tokens = np.asarray(self.data[idx], dtype=np.int64).copy()
        padding_mask = tokens == self.pad_id
        input_ids = tokens.copy()
        labels = np.full(tokens.shape, -100, dtype=np.int64)

        # Never predict or corrupt padding/special tokens.
        eligible = ~padding_mask
        for special_id in self.special_ids:
            eligible &= tokens != special_id

        # Training masks vary each access; validation masks are deterministic per block.
        if self.is_train:
            rng = np.random.default_rng()
        else:
            rng = np.random.default_rng(self.seed + int(idx))

        mask_indices = (rng.random(len(tokens)) < self.cfg.mask_prob) & eligible
        positions = np.flatnonzero(mask_indices)

        for j in positions:
            labels[j] = tokens[j]
            r = rng.random()
            if r < 0.80:
                input_ids[j] = self.cfg.mask_token_id
            elif r < 0.90:
                input_ids[j] = int(self.random_token_ids[rng.integers(0, len(self.random_token_ids))])
            # remaining 10%: retain original token

        return (
            torch.from_numpy(input_ids),
            torch.from_numpy(labels),
            torch.from_numpy(padding_mask.astype(np.bool_)),
        )

train_ds = MLMDataset(
    train_arr, CFG.max_seq_len, CFG, PAD_ID, BASE_SPECIAL_IDS,
    RANDOM_TOKEN_IDS, is_train=True, seed=CFG.seed
)
val_ds = MLMDataset(
    val_arr, CFG.max_seq_len, CFG, PAD_ID, BASE_SPECIAL_IDS,
    RANDOM_TOKEN_IDS, is_train=False, seed=CFG.seed + 100_000
)

# Seed sampler/workers for reproducibility; training masks still change each access.
torch_generator = torch.Generator()
torch_generator.manual_seed(CFG.seed)
train_dl = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2,
    pin_memory=(DEVICE.type == "cuda"), drop_last=True,
    generator=torch_generator, persistent_workers=True
)
val_dl = DataLoader(
    val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2,
    pin_memory=(DEVICE.type == "cuda"), drop_last=False,
    persistent_workers=True
)

print(f"Train: {len(train_ds):,} blocks ({len(train_ds)*CFG.max_seq_len:,} positions)")
print(f"Val:   {len(val_ds):,} blocks")
print(f"Steps/epoch ~ {len(train_ds) // CFG.batch_size // CFG.grad_accum:,}")

## 6 — Model architecture

Encoder-only Transformer with:
- **Full bidirectional self-attention** (no causal mask)
- **Rotary Positional Embeddings (RoPE)**
- **RMSNorm** (pre-norm)
- **SwiGLU** feed-forward
- **Weight tying** (embedding <-> MLM head)
- **MLM head**: Dense + GELU + RMSNorm + Linear

In [ ]:
# ── RMSNorm ──
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

# ── RoPE ──
def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    """x: (B, H, T, D)"""
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], dim=-1)
    return x * c_exp + rotated * s_exp

# ── Multi-Head Self-Attention (bidirectional) ──
class MultiHeadSelfAttention(nn.Module):
    """Full self-attention for encoder — NO causal mask."""
    def __init__(self, cfg):
        super().__init__()
        self.nh = cfg.n_heads
        self.hd = cfg.head_dim
        self.dm = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Full bidirectional attention
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if padding_mask is not None:
            w = w.masked_fill(
                padding_mask.unsqueeze(1).unsqueeze(2),
                float("-inf")
            )

        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))

# ── SwiGLU FFN ──
class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

# ── Encoder Block ──
class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)

    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        x = x + self.ffn(self.norm2(x))
        return x

# ── Full Encoder Model ──
class LegalEncoderLM(nn.Module):
    """Encoder-only Transformer with MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.layers  = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model)

        # MLM head: dense + activation + norm + projection
        self.mlm_dense = nn.Linear(cfg.d_model, cfg.d_model)
        self.mlm_norm  = RMSNorm(cfg.d_model)
        self.mlm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # Initialize independent modules first, then tie weights so the shared
        # embedding/output matrix is initialized exactly once.
        self.apply(self._init)
        if cfg.weight_tying:
            self.mlm_head.weight = self.tok_emb.weight

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None, padding_mask=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))

        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)

        x = self.norm(x)

        # MLM head
        h = F.gelu(self.mlm_dense(x))
        h = self.mlm_norm(h)
        logits = self.mlm_head(h)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, self.cfg.vocab_size),
                labels.view(-1),
                ignore_index=-100
            )
        return logits, loss

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def get_embeddings(self, input_ids, padding_mask=None):
        """Get contextual embeddings (for downstream tasks)."""
        was_training = self.training
        self.eval()
        x = self.tok_emb(input_ids)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        x = self.norm(x)
        if was_training:
            self.train()
        return x

model = LegalEncoderLM(CFG).to(DEVICE)
n_params = model.count_params()
print(f"Model: {n_params:,} parameters ({n_params/1e6:.1f}M)")
assert 50_000_000 <= n_params <= 300_000_000, f"Outside 50M-300M range: {n_params:,}"

# ── Smoke test ──
_x = torch.randint(0, CFG.vocab_size, (2, 32), device=DEVICE)
_y = torch.full((2, 32), -100, dtype=torch.long, device=DEVICE)
_y[:, 5] = _x[:, 5]
_logits, _loss = model(_x, labels=_y)
print(f"Smoke test: logits={_logits.shape}, loss={_loss.item():.4f}")
del _x, _y, _logits, _loss
torch.cuda.empty_cache()
print("Model OK")

## 7 — Training loop

In [ ]:
def get_lr(step):
    if step < CFG.warmup_steps:
        return CFG.lr * (step + 1) / max(1, CFG.warmup_steps)
    if step >= CFG.max_steps:
        return CFG.min_lr
    prog = (step - CFG.warmup_steps) / max(1, CFG.max_steps - CFG.warmup_steps)
    cos_val = 0.5 * (1.0 + math.cos(math.pi * prog))
    return CFG.min_lr + (CFG.lr - CFG.min_lr) * cos_val

@torch.no_grad()
def evaluate(model, dl, max_batches=None):
    """Token-weighted MLM loss. Validation masks are fixed by MLMDataset."""
    was_training = model.training
    model.eval()
    total_loss = 0.0
    total_targets = 0

    for i, (x, y, padding_mask) in enumerate(dl):
        if max_batches is not None and i >= max_batches:
            break
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        padding_mask = padding_mask.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(CFG.use_amp and DEVICE.type == "cuda")
        ):
            logits, _ = model(x, padding_mask=padding_mask)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, CFG.vocab_size).float(),
            y.reshape(-1),
            ignore_index=-100,
            reduction="sum",
        )
        n_targets = int((y != -100).sum().item())
        total_loss += float(loss_sum.item())
        total_targets += n_targets

    if was_training:
        model.train()
    if total_targets == 0:
        raise ValueError("Validation produced zero MLM targets. Check masking and labels.")
    return total_loss / total_targets

def save_ckpt(model, opt, scaler, step, val_loss, path, no_improve=0):
    """Atomic checkpoint write. val_loss is best-so-far for periodic checkpoints."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = str(path) + ".tmp"
    torch.save({
        "step": int(step),
        "val_loss": float(val_loss),
        "best_val": float(val_loss),
        "no_improve": int(no_improve),
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "scaler": scaler.state_dict(),
    }, tmp_path)
    os.replace(tmp_path, path)
    print(f"  Saved {path}")

# ── Build optimizer ──
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if p.ndim < 2 or "norm" in name.lower() or "bias" in name.lower():
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW([
    {"params": decay, "weight_decay": CFG.weight_decay},
    {"params": no_decay, "weight_decay": 0.0},
], lr=CFG.lr, betas=CFG.betas)

scaler = torch.amp.GradScaler("cuda", enabled=(CFG.use_amp and DEVICE.type == "cuda"))

# ── Resume from the latest checkpoint by step, not automatically from best.pt ──
start_step = 0
best_val = float("inf")
no_improve = 0
step_candidates = list(CKPT_DIR.glob("step_*.pt"))
# The step is encoded in step_N.pt, so choose the latest periodic checkpoint
# without loading potentially multi-gigabyte checkpoint files just to inspect them.
if step_candidates:
    resume = max(step_candidates, key=lambda path: int(path.stem.split("_")[1]))
elif (CKPT_DIR / "best.pt").exists():
    resume = CKPT_DIR / "best.pt"
else:
    resume = None

if resume is not None:
    try:
        ckpt = torch.load(resume, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["opt"])
        if "scaler" in ckpt:
            scaler.load_state_dict(ckpt["scaler"])
        start_step = int(ckpt["step"])
        best_val = float(ckpt.get("best_val", ckpt.get("val_loss", float("inf"))))
        no_improve = int(ckpt.get("no_improve", 0))
        print(f"Resumed from {resume.name}, step={start_step}, best_val={best_val:.4f}")
    except Exception as e:
        raise RuntimeError(
            f"Could not load checkpoint {resume}. If the vocabulary/model shape changed, "
            "move incompatible old checkpoints out of output/checkpoints and rerun."
        ) from e

print(f"\nTraining from step {start_step} to {CFG.max_steps}...")

In [ ]:
model.train()
train_iter = iter(train_dl)
running = 0.0
t0 = time.time()
last_log_step = start_step

for step in range(start_step, CFG.max_steps):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    acc_loss = 0.0

    for _ in range(CFG.grad_accum):
        try:
            x, y, padding_mask = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl)
            x, y, padding_mask = next(train_iter)

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        padding_mask = padding_mask.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(CFG.use_amp and DEVICE.type == "cuda")
        ):
            _, loss = model(x, labels=y, padding_mask=padding_mask)
            loss = loss / CFG.grad_accum

        scaler.scale(loss).backward()
        acc_loss += float(loss.item())

    scaler.unscale_(optimizer)
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
    scaler.step(optimizer)
    scaler.update()
    running += acc_loss

    if (step + 1) % CFG.log_interval == 0:
        elapsed = time.time() - t0
        n_steps = step + 1 - last_log_step
        avg_loss = running / max(n_steps, 1)
        tokens_per_sec = n_steps * CFG.eff_batch * CFG.max_seq_len / max(elapsed, 1e-9)
        print(
            f"step {step+1:6d}/{CFG.max_steps} | loss {avg_loss:.4f} | "
            f"lr {lr:.2e} | grad_norm {float(grad_norm):.2f} | "
            f"tok/s {tokens_per_sec:.0f} | {elapsed:.0f}s"
        )
        running = 0.0
        t0 = time.time()
        last_log_step = step + 1

    if (step + 1) % CFG.eval_interval == 0:
        vl = evaluate(model, val_dl, max_batches=50)
        print(f"  >>> validation MLM loss={vl:.4f}")

        if vl < best_val:
            best_val = vl
            no_improve = 0
            save_ckpt(
                model, optimizer, scaler, step + 1, best_val,
                CKPT_DIR / "best.pt", no_improve=no_improve
            )
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve}/{CFG.patience} evaluations")
            if no_improve >= CFG.patience:
                # Save the current stopping point for diagnosis/resume.
                save_ckpt(
                    model, optimizer, scaler, step + 1, best_val,
                    CKPT_DIR / f"step_{step+1}.pt", no_improve=no_improve
                )
                print(f"Early stopping at step {step+1}; best checkpoint remains best.pt")
                break
        model.train()

    if (step + 1) % CFG.save_interval == 0:
        save_ckpt(
            model, optimizer, scaler, step + 1, best_val,
            CKPT_DIR / f"step_{step+1}.pt", no_improve=no_improve
        )
else:
    # Normal completion only; this is the latest checkpoint, not necessarily best.
    save_ckpt(
        model, optimizer, scaler, CFG.max_steps, best_val,
        CKPT_DIR / "final.pt", no_improve=no_improve
    )

print(f"\nTraining stopped at step {step+1 if CFG.max_steps > start_step else start_step}.")
print(f"Best observed validation MLM loss: {best_val:.4f}")

## 8 — Load best checkpoint

In [ ]:

best_path = CKPT_DIR / "best.pt"
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded best checkpoint (step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f})")

    # Full validation pass for the final reported score (no batch limit).
    full_val_loss = evaluate(model, val_dl, max_batches=None)
    print(f"Full validation MLM loss: {full_val_loss:.4f}")
else:
    print("No best checkpoint found — using current model weights.")


## 9 — MLM Fill-in Demo

Since this is an **encoder** model, it doesn't generate text like the decoder.
Instead, it predicts masked tokens — useful for understanding, classification,
and downstream fine-tuning.

In [ ]:
model.eval()

def mlm_predict(text, mask_position, top_k=10):
    """Mask one token and predict top-k candidates."""
    ids = tokenizer.encode(text)
    if mask_position >= len(ids):
        print(f"Position {mask_position} out of range (len={len(ids)})")
        return
    original_id = ids[mask_position]
    original_word = tokenizer.decode([original_id])
    ids[mask_position] = CFG.mask_token_id

    inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits, _ = model(inp)

    probs = F.softmax(logits[0, mask_position], dim=-1)
    topk = torch.topk(probs, top_k)

    print(f"Original token: '{original_word}' (id={original_id})")
    print(f"Top-{top_k} predictions:")
    for i in range(top_k):
        tid = topk.indices[i].item()
        prob = topk.values[i].item()
        word = tokenizer.decode([tid])
        marker = " <--" if tid == original_id else ""
        print(f"  {i+1:2d}. '{word}' ({prob:.4f}){marker}")
    print()

# Demo prompts — mask different positions
demo_texts = [
    "The Supreme Court held that the appellant was entitled to relief under the Act.",
    "Section 302 of the Indian Penal Code provides that whoever commits murder shall be punished.",
    "The learned counsel for the respondent submitted that the lower court erred in law.",
]

for text in demo_texts:
    ids = tokenizer.encode(text)
    # Mask a token in the middle
    mid = len(ids) // 2
    print(f"TEXT: {text}")
    print(f"Masking position {mid}:")
    mlm_predict(text, mid, top_k=5)

## 10 — Done!

| File | Description |
|---|---|
| `output/checkpoints/best.pt` | Best model checkpoint (lowest val loss) |
| `output/checkpoints/final.pt` | Final model checkpoint |

The encoder model can be used for downstream tasks like:
- **Text classification** (judgment prediction)
- **Named entity recognition**
- **Semantic similarity**
- **Legal document retrieval**